# Opinionated Python 3.7+ IoC Proposal (Ports + Adapters, constructor-only)

**Goal:** a small, strict inversion-of-control (IoC) library for Python 3.7+ that enforces **constructor injection only** via `@Invert` on `__init__`, with implementations registered as `@Adapter(...)`.

Generated: `2026-01-29 09:54:53Z`


## Package / module layout (opinionated)

```
ioc/
  __init__.py       # re-exports (optional)
  errors.py         # typed exceptions
  scope.py          # ScopeEnum
  types.py          # typing helpers & small dataclasses
  env.py            # environment detection abstraction
  registry.py       # AdapterRegistry + metadata + keyed cache
  container.py      # Container / resolver: creates object graphs
  decorators.py     # class-based decorators: @Adapter, @Invert
```

## Core behavior rules

### Adapter selection precedence (for a given Port)
1. **Environment match wins**: if any registered Adapter has `env == current_env`, select it.
2. Else **default wins**: select the one registered with `default=True`.
3. Else if **only one Adapter exists**, use it.
4. Otherwise → `AmbiguousAdapterError`.

### Scopes
- `Singleton`: one instance reused (default policy: per-container singleton cache).
- `Unique`: new instance each resolution.
- `Keyed`: one instance per `(Port, key)`; entries may expire when `invalidate=True`.

### Keyed invalidation
- If `invalidate=True`, keyed entries are eligible for eviction after inactivity.
- Default idle timeout: **600 seconds (10 minutes)**.


## `ioc/errors.py`

In [ ]:
from typing import Optional, Type


class IoCError(Exception):
    """Base class for all IoC-related errors."""


class RegistrationError(IoCError):
    """Raised when adapter registration fails."""


class ResolutionError(IoCError):
    """Raised when dependency resolution fails."""


class DuplicateDefaultAdapterError(RegistrationError):
    """
    Raised when more than one Adapter is registered as default for a given Port.

    :param port: The Port type for which duplicates were detected.
    """
    def __init__(self, port: Type[object]) -> None:
        super().__init__()


class AmbiguousAdapterError(ResolutionError):
    """
    Raised when multiple candidate Adapters exist and no selection rule can choose one.

    :param port: The Port being resolved.
    :param env: The current environment name (if any).
    """
    def __init__(self, port: Type[object], env: Optional[str]) -> None:
        super().__init__()


class MissingAdapterError(ResolutionError):
    """
    Raised when no Adapter is registered for a required Port.

    :param port: The Port being resolved.
    :param env: The current environment name (if any).
    """
    def __init__(self, port: Type[object], env: Optional[str]) -> None:
        super().__init__()


class KeyRequiredError(ResolutionError):
    """
    Raised when resolving a Keyed adapter but no key is provided.

    :param port: The Port being resolved.
    """
    def __init__(self, port: Type[object]) -> None:
        super().__init__()


class InvalidAdapterError(RegistrationError):
    """
    Raised when an Adapter class is invalid for a Port.

    For example, the Adapter does not implement the Port, or is not instantiable.

    :param port: The Port type.
    :param adapter: The Adapter type.
    :param reason: Human-readable reason.
    """
    def __init__(self, port: Type[object], adapter: Type[object], reason: str) -> None:
        super().__init__()


## `ioc/scope.py`

In [ ]:
from enum import Enum


class ScopeEnum(str, Enum):
    """
    Supported lifecycle scopes for Adapters.

    - ``SINGLETON``: One instance reused.
    - ``UNIQUE``: A new instance created per resolution.
    - ``KEYED``: One instance per (Port, key).
    """
    SINGLETON = "Singleton"
    UNIQUE = "Unique"
    KEYED = "Keyed"


## `ioc/types.py`

In [ ]:
from dataclasses import dataclass
from typing import Any, Callable, Optional, Type

PortType = Type[object]
AdapterType = Type[object]
KeyType = str


@dataclass(frozen=True)
class ResolutionRequest:
    """
    A normalized request for resolving a Port.

    :param port: The Port type to resolve.
    :param env: The environment name in effect (e.g. ``dev`` / ``prod``).
    :param key: Optional key for keyed scope.
    """
    port: PortType
    env: Optional[str]
    key: Optional[KeyType]


class Container:
    """
    Forward declaration type for factories.

    The concrete Container lives in ``ioc/container.py``.
    """
    pass


AdapterFactory = Callable[[Container], Any]


## `ioc/env.py`

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Optional


class EnvironmentProvider(ABC):
    """
    Abstraction for reading the current environment.

    This exists so unit tests can override environment behavior cleanly.
    """

    @abstractmethod
    def get_env(self) -> Optional[str]:
        """
        Return the current environment name.

        :returns: Environment name (e.g. ``"dev"``) or ``None`` if not set.
        """
        raise NotImplementedError


@dataclass(frozen=True)
class OsEnvironProvider(EnvironmentProvider):
    """
    Reads the environment from an OS env var (default: ``PYTHON_ENV``).

    :param env_var: The OS environment variable name to read.
    """
    env_var: str = "PYTHON_ENV"

    def get_env(self) -> Optional[str]:
        """
        Return environment name from the configured OS env var.

        :returns: Environment string or ``None``.
        """
        raise NotImplementedError


## `ioc/registry.py`

In [ ]:
from dataclasses import dataclass
from threading import RLock
from typing import Any, Dict, List, Optional, Tuple

from .errors import (
    AmbiguousAdapterError,
    DuplicateDefaultAdapterError,
    InvalidAdapterError,
    MissingAdapterError,
)
from .scope import ScopeEnum
from .types import AdapterType, KeyType, PortType, ResolutionRequest


@dataclass(frozen=True)
class AdapterRegistration:
    """
    Metadata describing a registered Adapter implementation for a Port.

    :param port: The Port this Adapter satisfies.
    :param adapter: The Adapter concrete class.
    :param scope: Lifecycle scope for the Adapter.
    :param default: Whether this Adapter is the default choice for the Port.
    :param env: Optional environment name (e.g. ``dev``). If it matches, it wins.
    :param env_var: Name of env var consulted to determine current environment.
    :param key: Required if scope is ``KEYED``; identifies keyed instance bucket.
    :param invalidate: If True, keyed instances may expire after inactivity timeout.
    """
    port: PortType
    adapter: AdapterType
    scope: ScopeEnum
    default: bool
    env: Optional[str]
    env_var: str
    key: Optional[KeyType]
    invalidate: bool


@dataclass
class KeyedEntry:
    """
    An entry in the keyed instance cache.

    :param instance: The cached Adapter instance.
    :param last_used_epoch: Seconds since epoch when instance was last accessed.
    :param invalidate: Whether this entry can expire.
    """
    instance: Any
    last_used_epoch: float
    invalidate: bool

    def touch(self) -> None:
        """
        Update the `last_used_epoch` timestamp to now.
        """
        raise NotImplementedError


class AdapterRegistry:
    """
    Central registry of Port -> Adapter registrations and keyed instance cache.

    This class is responsible for:
    - Registering Adapters declared via `@Adapter(...)`
    - Enforcing default uniqueness per Port
    - Selecting adapters using env/default rules
    - Caching keyed instances (optional eviction)

    Thread-safety:
      - Uses an internal re-entrant lock.
      - Registration and cache mutations are protected.
    """

    def __init__(self, keyed_idle_ttl_seconds: int = 600) -> None:
        """
        :param keyed_idle_ttl_seconds: Default idle timeout for invalidatable keyed entries.
        """
        self._lock: RLock = RLock()
        self._keyed_idle_ttl_seconds: int = keyed_idle_ttl_seconds

        # Port -> list[AdapterRegistration]
        self._registrations: Dict[PortType, List[AdapterRegistration]] = {}

        # (Port, key) -> KeyedEntry
        self._keyed_instances: Dict[Tuple[PortType, KeyType], KeyedEntry] = {}

    def register(self, reg: AdapterRegistration) -> None:
        """
        Register an Adapter for a Port.

        Enforces:
        - If `reg.default` is True, only one default Adapter may exist per Port.

        :param reg: Registration metadata to add.
        :raises DuplicateDefaultAdapterError: If a second default is registered for the Port.
        :raises InvalidAdapterError: If the Adapter is invalid for the Port.
        """
        raise NotImplementedError

    def list_ports(self) -> List[PortType]:
        """
        Return all Ports with any registrations.

        :returns: List of Port types.
        """
        raise NotImplementedError

    def get_registrations(self, port: PortType) -> List[AdapterRegistration]:
        """
        Return all registered adapters for a Port.

        :param port: The Port type.
        :returns: Registrations for the Port (possibly empty).
        """
        raise NotImplementedError

    def select(self, request: ResolutionRequest) -> AdapterRegistration:
        """
        Select the best AdapterRegistration for a given resolution request.

        Selection rules:
        1) env match wins
        2) default wins
        3) if only one exists, it wins
        else ambiguous

        :param request: Normalized request.
        :returns: Selected AdapterRegistration.
        :raises MissingAdapterError: If no candidates exist.
        :raises AmbiguousAdapterError: If no unique winner exists.
        """
        raise NotImplementedError

    def get_keyed_entry(self, port: PortType, key: KeyType) -> Optional[KeyedEntry]:
        """
        Retrieve a keyed cache entry if present.

        :param port: The Port type.
        :param key: The keyed instance key.
        :returns: Entry if present, else None.
        """
        raise NotImplementedError

    def set_keyed_entry(self, port: PortType, key: KeyType, entry: KeyedEntry) -> None:
        """
        Store a keyed cache entry (overwriting any prior entry).

        :param port: The Port type.
        :param key: The instance key.
        :param entry: The cache entry.
        """
        raise NotImplementedError

    def prune_keyed(self, now_epoch: Optional[float] = None) -> int:
        """
        Remove expired keyed entries.

        An entry is expired if:
        - entry.invalidate is True
        - and (now - entry.last_used_epoch) >= ttl

        :param now_epoch: Override current time for deterministic tests.
        :returns: Number of entries removed.
        """
        raise NotImplementedError

    def keyed_idle_ttl_seconds(self) -> int:
        """
        Return the current default TTL for invalidatable keyed entries.

        :returns: TTL seconds.
        """
        raise NotImplementedError


## `ioc/container.py`

In [ ]:
from dataclasses import dataclass
from enum import Enum
import inspect
from typing import Any, Dict, Optional, Type, TypeVar

from .env import EnvironmentProvider, OsEnvironProvider
from .errors import AmbiguousAdapterError, KeyRequiredError, MissingAdapterError
from .registry import AdapterRegistry
from .scope import ScopeEnum
from .types import KeyType, PortType, ResolutionRequest

T = TypeVar("T")


class ContainerScopePolicy(str, Enum):
    """
    Controls where singleton instances live.

    - ``PER_CONTAINER``: Singleton cache is per Container instance.
    - ``GLOBAL``: Singleton cache is global/shared via the registry (less test-friendly).
    """
    PER_CONTAINER = "PerContainer"
    GLOBAL = "Global"


@dataclass
class ContainerConfig:
    """
    Configuration for the Container.

    :param registry: AdapterRegistry holding registrations and keyed cache.
    :param env_provider: Provider for current environment string.
    :param scope_policy: Policy controlling singleton cache behavior.
    """
    registry: AdapterRegistry
    env_provider: EnvironmentProvider = OsEnvironProvider()
    scope_policy: ContainerScopePolicy = ContainerScopePolicy.PER_CONTAINER


class Container:
    """
    Resolver that builds object graphs using constructor injection.

    Rules:
    - Constructor-only injection
    - Port types are determined from `__init__` annotations
    - Concrete Adapter is selected by the registry using env/default rules
    - Adapter lifecycle is enforced via ScopeEnum

    This Container is intentionally opinionated:
    - If a parameter is not annotated, it is not injected.
    - If a parameter has a default value and no injection is requested, default applies.
    """

    def __init__(self, config: ContainerConfig) -> None:
        """
        :param config: Container configuration.
        """
        self._config: ContainerConfig = config
        self._singletons: Dict[PortType, Any] = {}

    def env(self) -> Optional[str]:
        """
        Return current environment name.

        :returns: Environment string or None.
        """
        raise NotImplementedError

    def resolve(self, port: PortType, key: Optional[KeyType] = None) -> Any:
        """
        Resolve an instance for a Port.

        :param port: The Port type to resolve.
        :param key: Optional key for keyed scope.
        :returns: Instance implementing the Port.
        :raises MissingAdapterError: If no adapter exists.
        :raises AmbiguousAdapterError: If multiple candidates exist with no winner.
        :raises KeyRequiredError: If the chosen scope is keyed but no key was provided.
        """
        raise NotImplementedError

    def create(self, cls: Type[T], **overrides: Any) -> T:
        """
        Create an instance of `cls`, injecting constructor dependencies.

        Overrides:
        - Any keyword arguments provided in `overrides` are used as-is
          and are not injected.

        :param cls: Concrete class to instantiate.
        :param overrides: Explicit constructor arguments.
        :returns: New instance of `cls`.
        """
        raise NotImplementedError

    def _build_constructor_kwargs(self, cls: Type[T], overrides: Dict[str, Any]) -> Dict[str, Any]:
        """
        Inspect `cls.__init__` signature and build kwargs for injection.

        :param cls: Class to construct.
        :param overrides: Explicit args supplied by caller.
        :returns: Keyword args to pass to constructor.
        """
        raise NotImplementedError

    def _resolve_via_registration(self, port: PortType, key: Optional[KeyType]) -> Any:
        """
        Resolve by selecting an AdapterRegistration and applying scope policy.

        :param port: Port to resolve.
        :param key: Optional key for keyed scope.
        :returns: Instance.
        """
        raise NotImplementedError


## `ioc/decorators.py`

In [ ]:
from dataclasses import dataclass
from typing import Any, Callable, Dict, Optional, Type, TypeVar

from .container import Container
from .errors import InvalidAdapterError
from .registry import AdapterRegistration, AdapterRegistry
from .scope import ScopeEnum
from .types import KeyType, PortType

T = TypeVar("T")


class ContainerProvider:
    """
    Stores and exposes a default Container for `@Invert` usage.

    This avoids global free-functions while still allowing a simple default path.
    """

    _container: Optional[Container] = None

    @classmethod
    def set_default(cls, container: Container) -> None:
        """
        Set the default container used by `@Invert`.

        :param container: Container to set.
        """
        cls._container = container

    @classmethod
    def get_default(cls) -> Container:
        """
        Return the default container.

        :returns: Container instance.
        :raises RuntimeError: If no default container has been set.
        """
        raise NotImplementedError


@dataclass(frozen=True)
class InvertKeyMap:
    """
    Maps constructor parameter names to keys for keyed resolution.

    Example:
      ``InvertKeyMap({"tenant_db": "tenant-123"})``

    :param keys: Map from parameter name -> key string.
    """
    keys: Dict[str, KeyType]


class Adapter:
    """
    Class-based decorator used on Adapter classes to register them.

    Signature (as requested):

    .. code-block:: python

        @Adapter(scope=ScopeEnum.SINGLETON, default=True, env=None,
                env_var="PYTHON_ENV", key=None, invalidate=True)
        class SomeAdapter(SomePort):
            ...

    Port inference:
    - Prefer `__ports__` tuple on the Adapter class if present
    - Else infer from base classes that look like Ports

    Registration target:
    - Uses the configured AdapterRegistry instance.
    - By default, uses the global/default registry set via `Adapter.configure_registry`.
    """

    _registry: Optional[AdapterRegistry] = None

    @classmethod
    def configure_registry(cls, registry: AdapterRegistry) -> None:
        """
        Set the registry used by this decorator.

        :param registry: AdapterRegistry to use.
        """
        cls._registry = registry

    def __init__(
        self,
        scope: ScopeEnum,
        default: bool = True,
        env: Optional[str] = None,
        env_var: str = "PYTHON_ENV",
        key: Optional[str] = None,
        invalidate: bool = True,
    ) -> None:
        """
        :param scope: Adapter scope (Singleton, Unique, Keyed).
        :param default: Whether this adapter is the default for its Port(s).
        :param env: Environment name that, if matched, overrides default selection.
        :param env_var: Environment variable name used to determine current env.
        :param key: Key string required if scope is Keyed.
        :param invalidate: If True, keyed entries may expire after inactivity timeout.
        """
        self._scope = scope
        self._default = default
        self._env = env
        self._env_var = env_var
        self._key = key
        self._invalidate = invalidate

    def __call__(self, adapter_cls: Type[T]) -> Type[T]:
        """
        Decorate and register an Adapter class.

        :param adapter_cls: The concrete Adapter class.
        :returns: The same class, unmodified (registration side effect).
        :raises InvalidAdapterError: If ports cannot be inferred or adapter is invalid.
        """
        raise NotImplementedError


class Invert:
    """
    Class-based decorator used on `__init__` to enable constructor injection.

    Typical usage:

    .. code-block:: python

        class Service:
            @Invert()
            def __init__(self, repo: RepoPort, clock: ClockPort) -> None:
                ...

    Keyed injection:
    - Provide an `InvertKeyMap` mapping parameter name -> key:

    .. code-block:: python

        class MultiTenantThing:
            @Invert(keys=InvertKeyMap({"db": "tenant-123"}))
            def __init__(self, db: DatabasePort) -> None:
                ...

    Resolution behavior:
    - If caller provides the argument explicitly, `@Invert` does not override it.
    - Otherwise it resolves using `ContainerProvider.get_default()`.
    """

    def __init__(self, keys: Optional[InvertKeyMap] = None) -> None:
        """
        :param keys: Optional mapping to provide keyed resolution keys.
        """
        self._keys = keys

    def __call__(self, init: Callable[..., Any]) -> Callable[..., Any]:
        """
        Wrap `__init__` to inject missing annotated parameters from the container.

        :param init: The original `__init__` method.
        :returns: Wrapped `__init__`.
        """
        raise NotImplementedError


## Dependency chart (modules as boxes with class lanes)

This cell draws a module-level dependency diagram where:
- Each **module** is a box.
- Each **class** is a horizontal lane inside its module box.
- **Dependents (Ca)** conceptually attach to the **left** edge of a module.
- **Dependencies (Ce)** attach to the **right** edge of a module.
- Horizontal ordering is ranked by **Instability**: `I = Ce / (Ca + Ce)`  
  (more unstable modules placed left; more stable modules placed right).


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch

# A -> B means A depends on B (Ce edges)
deps = {
    "ioc.errors": set(),
    "ioc.scope": set(),
    "ioc.types": set(),
    "ioc.env": set(),
    "ioc.registry": {"ioc.errors", "ioc.scope", "ioc.types"},
    "ioc.container": {"ioc.env", "ioc.registry", "ioc.errors", "ioc.types", "ioc.scope"},
    "ioc.decorators": {"ioc.registry", "ioc.container", "ioc.errors", "ioc.scope", "ioc.types"},
}

# Classes per module (lanes)
classes = {
    "ioc.errors": ["IoCError", "RegistrationError", "ResolutionError",
                   "DuplicateDefaultAdapterError", "AmbiguousAdapterError",
                   "MissingAdapterError", "KeyRequiredError", "InvalidAdapterError"],
    "ioc.scope": ["ScopeEnum"],
    "ioc.types": ["ResolutionRequest", "Container", "AdapterFactory"],
    "ioc.env": ["EnvironmentProvider", "OsEnvironProvider"],
    "ioc.registry": ["AdapterRegistration", "KeyedEntry", "AdapterRegistry"],
    "ioc.container": ["ContainerScopePolicy", "ContainerConfig", "Container"],
    "ioc.decorators": ["ContainerProvider", "InvertKeyMap", "Adapter", "Invert"],
}

# Compute Ca, Ce, Instability
modules = list(deps.keys())
Ce = {m: len(deps[m]) for m in modules}
Ca = {m: 0 for m in modules}
for a, bs in deps.items():
    for b in bs:
        Ca[b] += 1

I = {}
for m in modules:
    denom = Ca[m] + Ce[m]
    I[m] = (Ce[m] / denom) if denom else 0.0

# Sort left->right: high instability to low instability
modules_sorted = sorted(modules, key=lambda m: (-I[m], m))

# Layout
box_w = 2.9
box_h_base = 0.6
lane_h = 0.28
x_gap = 0.8
y0 = 0.6

positions = {}
x = 0.6
for m in modules_sorted:
    h = box_h_base + lane_h * len(classes[m])
    positions[m] = (x, y0, box_w, h)
    x += box_w + x_gap

plt.figure(figsize=(18, 6))
ax = plt.gca()
ax.set_axis_off()

# Draw boxes + lanes
for m in modules_sorted:
    x, y, w, h = positions[m]
    ax.add_patch(Rectangle((x, y), w, h, fill=False, linewidth=1.2))
    header = f"{m}  |  Ca={Ca[m]}  Ce={Ce[m]}  I={I[m]:.2f}"
    ax.text(x + 0.08, y + h - 0.22, header, fontsize=9, va="top")

    lane_y = y + h - 0.55
    for cls in classes[m]:
        ax.add_patch(Rectangle((x + 0.06, lane_y - lane_h + 0.02), w - 0.12, lane_h, fill=False, linewidth=0.6))
        ax.text(x + 0.12, lane_y - 0.07, cls, fontsize=8, va="top")
        lane_y -= lane_h

def mid_y(mname: str) -> float:
    x, y, w, h = positions[mname]
    return y + h / 2.0

# Draw arrows: right edge of A -> left edge of B
for a, bs in deps.items():
    for b in bs:
        ax_x, ax_y, aw, ah = positions[a]
        bx_x, bx_y, bw, bh = positions[b]
        start = (ax_x + aw, mid_y(a))
        end = (bx_x, mid_y(b))
        # gentle curvature based on relative order
        rad = 0.10 * (modules_sorted.index(a) - modules_sorted.index(b))
        rad = max(-0.35, min(0.35, rad))
        arrow = FancyArrowPatch(
            start, end,
            arrowstyle="->",
            mutation_scale=10,
            linewidth=0.9,
            connectionstyle=f"arc3,rad={rad}",
        )
        ax.add_patch(arrow)

xmin = min(positions[m][0] for m in modules_sorted) - 0.4
xmax = max(positions[m][0] + positions[m][2] for m in modules_sorted) + 0.4
ymax = max(positions[m][1] + positions[m][3] for m in modules_sorted) + 0.4
ax.set_xlim(xmin, xmax)
ax.set_ylim(0, ymax)

plt.title("IoC Proposal: Module/Class Lanes with Dependencies (ranked by Instability)", fontsize=12)
plt.show()
